# Shut the box
---
<img src='https://www.janestreet.com/puzzles/shut-the-box.png' width=600 align='center'/>


Use a scissors to cut away one ore more groups of orthogonally connected cells (squares) from the grid above. Any group you cut away must have at least one cell along the boundary of the grid. The remaining cells must be orthogonally connected and not have any holes.

It must be possible to fold along some of the grid lines so that the remaining cells form the six walls of a rectangular solid (the “box”). There may not be any overlapping cells in the box.

Some cells have been labeled with arrows. These cells are not part of the box, but instead point in the direction(s) of the nearest box cells (looking in that square’s row and column).

Some cells have been labeled with numbers. These cells are part of the box. A number indicates how many cells within one king’s move of that cell are a part of the box. (Including the numbered cell.)

When the box is assembled, each grey circle should be directly opposite (1) another grey circle. Each gray square should be orthogonally adjacent to (and on the same face as) another gray square.

Once you have assembled the box, compute, on each face, the sum of the numbered cells. The answer to this puzzle is the product of these six sums.

1. (That is, the line segment connecting “opposite” circles should be orthogonal to the faces containing them.)

# The approach
---
This puzzle was solved in a group of 2 in a two step fashion. Cells belonging to the cuboid will be hereafter called net cells.

1. The grid rules constrain the net cells from the cutout cells in a sudoku-like fashion (likely solvable through a csp solver but I am not familiar with these **yet**), therefore you can reason your way into the following grid. White cells are ambiguous due to the grid being underconstrained.

<img src='boxdoku.webp' width=400 />

2. Once here, there are two valid approaches:
    - You can cut out the net and feel your way into a solution
    - You can enumerate the number of valid net combinations then see if a folding exists using O'Rourke and Lubiw's folding algorithm [[1](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=AdrTURsAAAAJ&cstart=200&pagesize=100&sortby=pubdate&citation_for_view=AdrTURsAAAAJ:IWHjjKOFINEC)] that tells you if a geometry is foldable and if so, how the edges connect to each other; these edge instructions guide you on how to fold the cuboid

The remaining of the write-up will focus on the latter.

# Dynamic programming algorithm intuition
---
Every time an edge-to-edge connection is made (green to green), a partition is made on the perimeter creating two subproblems (red and blue). See below for a visual [[1](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=AdrTURsAAAAJ&cstart=200&pagesize=100&sortby=pubdate&citation_for_view=AdrTURsAAAAJ:IWHjjKOFINEC)] using a latin cross polygon.

<img src='latin-cross.png' width=200 />

For each of these sub-perimeters, further partitions can then be made in a top-down fashion by connecting edges until you reach a base case where no more partitions can be made. Alternatively, you can do it in a bottoms-up fashion where you start with the smallest non-base case partitions and build up from there until you arrive at the original problem you're trying to solve. For a detailed description of the algorithm, see [[1](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=AdrTURsAAAAJ&cstart=200&pagesize=100&sortby=pubdate&citation_for_view=AdrTURsAAAAJ:IWHjjKOFINEC)].



In [1]:
#Bottoms-up dynamic programming implementation. Note we don't check for edge length mismatches because for this puzzle all nets have equal length edges.
class FoldingAlgo:
    def __init__(self, angles: list[int]) -> None:
        #For equal length edge nets, geometries can be defined via the inside angle at each vertex
        #I like to start at the bottom left of the polygon and move counter-clockwise
        self.angles = angles
        
        #Used for index modulo arithmetic
        self.n = len(angles)

        #Hashmap containing the minimum extra angle 
        self.d = {}

        #Hashmap tracking the folding which minimises the angle per each perimeter subpartition (needed to track instructions)
        self.f = {}

        #Instructions list
        self.instructions = []
    
    def solver(self) -> None:
        #Memoization of subproblems
        for i in range(self.n):
            a = 0
            b = i
            for _ in range(self.n):
                self.d[(a % self.n, b % self.n)] = self.condChecker(a, b)
                a += 1
                b += 1

        e0 = 0
        en = self.n
        
        for k in range(e0 + 1, en, 2):
            if self.validGluing(e0+1, k) and self.validGluing((k + 1) % self.n, e0):
                self.instructions.append(f"Glue edge {e0} with {k}")
                self.recur_fn(e0+1 % self.n, k)
                self.recur_fn((k+1) % self.n, e0)

                break


    #Function used to trace back the gluings that lead up to the original problem solving
    def recur_fn(self, x: int, y: int, visited = None) -> None:
        if visited is None:
            visited = set()

        edge = tuple(sorted((x, y)))
        if x == y or edge in visited:
            return

        #This is necessary to prevent recursive cycles
        visited.add(edge)

        #Reconstructing instructions
        self.instructions.append(self.f[(x % self.n,y % self.n)][1])
        b1, b2 = self.f[(x % self.n, y % self.n)][0]

        self.recur_fn((b1 + 1) % self.n, b2, visited = visited)
        self.recur_fn((b2 + 1) % self.n, b1, visited = visited)


    def validGluing(self, x: int, y: int) -> bool:
        if x == y:
            return True
        
        else:
            return self.angles[x] + self.angles[y] + self.d[(x,y)] <= 360


    '''
    Checks the first D3 condition
    IMPORTANT
    For nets where each cell represents a face on the folded polyhedron, the first constraint must require the vertex curvature be < 360. 
    For a cuboid polyhedron, it's sufficient that the vertex curvature be ≤ 360 as the polyhedron faces can be several vertexes long allowing for "flat" vertices.
    '''
    def firstChecker(self, x: int, y: int) -> bool:
        
        if self.angles[x % self.n] + self.angles[y % self.n] + self.d[(x % self.n, y % self.n)] > 360:
            return False

        else:
            return True
        
    '''
    Checks the second D3 condition
    '''
    def secondChecker(self, x: int, y: int) -> int:
        if x == y:
            return 0
        
        else:
            return self.angles[x % self.n] + self.d[(x % self.n, y % self.n)]

    #Logic for condition checking and for tracking instructions
    def condChecker(self, a: int, b: int) -> int:
        #print(f"a and b are: {a}, {b}")
        if a == b:
            return 0
        
        if abs(a - b) % 2 != 0:
            return float('inf')

        else:
            extra = float('inf')
            for k in range(a+1, b, 2):

                if a + 1 == k:
                    c = self.secondChecker(k+1, b)
                    extra = min(extra, c)

                    #If this combination minimises the extra angle, store it for reconstruction purposes
                    if c <= extra:
                        self.f[(a,b % self.n)] = [(a, k % self.n), f"Glue edge {a} with {k % self.n}"]
                    continue
                
                elif self.firstChecker(a+1, k):
                    c = self.secondChecker(k+1, b)

                    #If this combination minimises the extra angle, store it for reconstruction purposes
                    if c < extra:
                        self.f[(a,b % self.n)] = [(a, k % self.n), f"Glue edge {a} with {k % self.n}"]
                    extra = min(extra, c)

                else:
                    extra = min(extra, float('inf'))
                    
            return extra

## Solved example
<img src="solved_puzzle.png" width=500/>

To succesfully verify the algorithm, we used the the provided worked out example.

In [2]:
#Vertex angles are counted counterclockwise starting at the bottom leftmost one 
#This is an encoding of the geometry from the worked out example
solved = [
        90, 180, 180, 180, 90, 270, 270, 90, 
        90, 270, 90, 90, 180, 180, 270,
        180, 180, 270, 270, 90, 180, 180, 90,
        90, 180, 270, 90, 180, 270, 90, 180, 90,
        270, 180, 90, 90, 180, 270, 270, 180, 90,
        90, 180, 180, 270, 270, 90, 270, 180, 90
]

a = FoldingAlgo(solved)
a.solver()

for line in a.instructions:
    print(f"{line}")

Glue edge 0 with 43
Glue edge 1 with 42
Glue edge 2 with 41
Glue edge 3 with 6
Glue edge 4 with 5
Glue edge 7 with 40
Glue edge 8 with 19
Glue edge 9 with 18
Glue edge 10 with 17
Glue edge 11 with 16
Glue edge 12 with 15
Glue edge 13 with 14
Glue edge 20 with 39
Glue edge 21 with 38
Glue edge 22 with 37
Glue edge 23 with 36
Glue edge 24 with 35
Glue edge 25 with 34
Glue edge 26 with 29
Glue edge 27 with 28
Glue edge 30 with 33
Glue edge 31 with 32
Glue edge 34 with 25
Glue edge 40 with 7
Glue edge 41 with 2
Glue edge 44 with 49
Glue edge 45 with 48
Glue edge 46 with 47


## Puzzle question
Looking once again at the "sudoku" solution, all that's left to do is encode all the valid variations with the ambiguous squares and see which one folds and how.

For brevity we only include the net the leads to the solution.

<img src='boxdoku2.png' width=400 />







In [3]:
#Encoding of the puzzle question
puzzle = [
    90, 90, 270, 90, 90, 270, 270, 180, 270, 90, 270, 90, 90,
    270, 90, 90, 180, 270, 180, 90, 180, 90, 270, 180, 270, 90,
    270, 270, 180, 90, 90, 180, 270, 180, 270, 270, 90, 90, 180,
    90, 270, 90, 270, 270, 90, 270, 90, 90, 270, 270, 180, 90,
    270, 270, 180, 270, 90, 180, 180, 180, 180, 180, 180, 180,
    90, 90, 270, 90, 180, 90, 270, 180, 270, 270, 90, 180, 90,
    270, 90, 90, 180, 270, 90, 270, 270, 90, 180, 270, 90, 270,
    90, 270, 270, 180, 180, 180, 180, 180, 180, 180, 270, 90,
    180, 90, 90, 270, 180, 90, 90, 270, 270, 90, 270, 90, 90,
    270, 90, 270, 180, 270, 270, 90, 90, 180, 90, 180, 270,
    270, 180, 90, 90, 270, 90, 180, 90, 270, 180, 90, 270,
    90, 270, 270, 90, 270, 90, 180, 90, 90, 270, 270, 90,
    270, 270, 90, 270, 270, 90, 90, 270, 90, 180, 90, 90,
    270, 180, 180, 90, 270, 270, 180, 90, 180, 270, 180,
    180, 180, 90, 270, 90, 90, 270, 90, 180, 90, 180,
    270, 180, 270, 270, 90, 180, 90, 180, 270, 180, 270,
    90, 270, 270, 90, 270, 180, 90, 180, 90, 90, 270, 180,
    180, 90, 180, 180, 270, 90, 180, 270, 90, 270, 270, 90, 90, 270
]

b = FoldingAlgo(puzzle)
b.solver()

for line in b.instructions:
    print(f"{line}")


Glue edge 0 with 151
Glue edge 1 with 150
Glue edge 2 with 149
Glue edge 3 with 148
Glue edge 4 with 147
Glue edge 5 with 146
Glue edge 6 with 145
Glue edge 7 with 144
Glue edge 8 with 143
Glue edge 9 with 142
Glue edge 10 with 141
Glue edge 11 with 140
Glue edge 12 with 139
Glue edge 13 with 138
Glue edge 14 with 137
Glue edge 15 with 18
Glue edge 16 with 17
Glue edge 19 with 136
Glue edge 20 with 135
Glue edge 21 with 134
Glue edge 22 with 133
Glue edge 23 with 132
Glue edge 24 with 131
Glue edge 25 with 130
Glue edge 26 with 129
Glue edge 27 with 128
Glue edge 28 with 127
Glue edge 29 with 126
Glue edge 30 with 125
Glue edge 31 with 124
Glue edge 32 with 123
Glue edge 33 with 122
Glue edge 34 with 121
Glue edge 35 with 120
Glue edge 36 with 119
Glue edge 37 with 118
Glue edge 38 with 117
Glue edge 39 with 116
Glue edge 40 with 115
Glue edge 41 with 114
Glue edge 42 with 113
Glue edge 43 with 112
Glue edge 44 with 111
Glue edge 45 with 110
Glue edge 46 with 109
Glue edge 47 with 108


Folding up the net using the instructions yields a sum product of 28x57x11x17x5x11 = 16,414,860